# Cell1 自检

自动断言 cache、AMP、num_workers、batch 三类别、NaN/Inf、真实幅值和系统内存。

In [1]:
from pathlib import Path
import importlib
import sys

RUN_DIR = Path.cwd() / 'experiments/cgan_v1/runs/2026-06-07_01_phase2a_full_newcache'
TRAIN_DIR = RUN_DIR / '02_train'
if str(TRAIN_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_DIR))

import train
importlib.reload(train)

config = train.load_config()
cell1_report = train.cell1_self_check(config)
cell1_report

{'timestamp': '2026-06-08 00:42:13 +0800',
 'cache_dir': '/home/liujia/rf_training_cache/cgan_phase2a_64x32x32_random_full1500_fp16_cache_20260607',
 'use_amp': False,
 'num_workers': 0,
 'loss_lambdas': {'lambda_adv': 1.0,
  'lambda_struct': 1.0,
  'lambda_carrier': 0.5},
 'struct_normalization': {'mode': 'label_mean_envelope', 'eps': 1e-08},
 'available_memory_gb': 58.42288589477539,
 'batch_categories': ['carotid', 'muscle', 'phantom'],
 'batch_stats': {'input': {'shape': [3, 1536, 64, 32, 32],
   'dtype': 'torch.float32',
   'max_abs': 10619.09375,
   'mean': -0.020570039749145508,
   'mean_abs': 261.2459716796875,
   'has_nan': False,
   'has_inf': False},
  'label': {'shape': [3, 2, 64, 32, 32],
   'dtype': 'torch.float32',
   'max_abs': 21726.177734375,
   'mean': -3.3417727947235107,
   'mean_abs': 2217.3935546875,
   'has_nan': False,
   'has_inf': False},
  'baseline': {'shape': [3, 2, 64, 32, 32],
   'dtype': 'torch.float32',
   'max_abs': 48332.2578125,
   'mean': -5.262175

# Cell2 配置

打印本次运行的唯一配置源摘要。

In [2]:
cell2_report = train.cell2_config(config)
cell2_report

{'run_name': '2026-06-08_01_phase2a_normstruct_v1',
 'cache_dir': '/home/liujia/rf_training_cache/cgan_phase2a_64x32x32_random_full1500_fp16_cache_20260607',
 'epochs': 50,
 'use_amp': False,
 'num_workers': 0,
 'loss': {'type': 'lsgan_struct_carrier',
  'lambda_adv': 1.0,
  'lambda_struct': 1.0,
  'lambda_carrier': 0.5,
  'struct_normalization': {'mode': 'label_mean_envelope',
   'eps': 1e-08,
   'scale_definition': 's = mean(abs(env(label))) + eps',
   'constraint': 'pred and label share the same label-derived scalar scale'},
  'lowpass': {'type': 'anisotropic_gaussian_3d',
   'sigma_voxels': {'z': 6.0, 'x': 3.0, 'y': 3.0},
   'truncate': 3.0}},
 'sampler': {'type': 'stratified_category_batch_sampler',
  'samples_per_category': 1,
  'effective_batch_size': 3,
  'shuffle': True,
  'seed': 20260607,
  'purpose': 'keep carotid/muscle/phantom present in each batch for BatchNorm3d'}}

# Cell3 数据

构建 train/val dataset、分层 batch sampler、固定 val 探针索引。

In [3]:
data_state = train.cell3_data(config)
data_state['summary']

{'train_samples': 1050, 'val_samples': 225}

# Cell4 模型

构建 TinyResidualRFNet(BatchNorm3d)、Envelope2DPatchDiscriminator、lowpass 和 optimizer。

In [4]:
model_state = train.cell4_models(config)
model_state['summary']

{'device': 'cuda',
 'G_params': 264738,
 'D_params': 662721,
 'G_first_norm': 'BatchNorm3d'}

# Cell5 训练循环

按 config 跑 50 epoch；每 epoch 写 stability.csv 和 speckle_check.csv，snapshot epoch 写三联探针图和 checkpoint。

In [5]:
train_result = train.cell5_train_loop(config, data_state, model_state)
train_result['result']

STRUCT_NORMALIZATION_SELF_CHECK old=3466.09 scale=4275.59 new=0.81067 old/scale=0.81067 rel_err=3.01e-08 adv_w=1.49114 struct_w=0.81067 carrier_w=2659.16 total=2661.46


epoch=001 batch=0001 D_real=1.46514 D_fake=0.151855 G_adv_w=0.248689 G_struct_w=0.81067 G_carrier_w=2659.16 G_total=2660.22 pred_max_abs=76592.8


epoch=001 batch=0050 D_real=0.115848 D_fake=0.0854606 G_adv_w=1.25532 G_struct_w=1.12433 G_carrier_w=2210.54 G_total=2212.92 pred_max_abs=36923.1


epoch=001 batch=0100 D_real=0.0419933 D_fake=0.0773175 G_adv_w=1.1392 G_struct_w=0.915168 G_carrier_w=950.322 G_total=952.376 pred_max_abs=18961.3


epoch=001 batch=0150 D_real=0.0667006 D_fake=0.0374616 G_adv_w=0.744425 G_struct_w=0.90494 G_carrier_w=1506.04 G_total=1507.69 pred_max_abs=38972.1


...

epoch=050 batch=0250 D_real=0.000179707 D_fake=0.000206475 G_adv_w=1.01071 G_struct_w=0.928996 G_carrier_w=6377.04 G_total=6378.98 pred_max_abs=161304


epoch=050 batch=0300 D_real=0.000196384 D_fake=0.000104138 G_adv_w=0.98806 G_struct_w=0.619143 G_carrier_w=2827.06 G_total=2828.67 pred_max_abs=108230


epoch=050 batch=0350 D_real=0.000399008 D_fake=0.000194741 G_adv_w=1.01318 G_struct_w=0.92818 G_carrier_w=3202.33 G_total=3204.27 pred_max_abs=76096.7


EPOCH_SUMMARY epoch=050 batches=350 D_real=0.000950661 D_fake=0.000915686 D_real_score=0.730845 D_fake_score=0.500208 G_adv_w=1.00069 G_struct_w=0.982056 G_carrier_w=3049.25 G_total=3051.23 pred_max_abs=280030 peakGB=2.608 memGB=52.792


{'timestamp': '2026-06-08 06:58:32 +0800',
 'status': 'completed',
 'abort_reason': None,
 'epochs_completed': 50,
 'epochs_expected': 50,
 'elapsed_sec': 22577.050738096237,
 'last_stability': {'epoch': 50,
  'batches': 350,
  'seconds': 445.492614030838,
  'peak_cuda_mem_gb': 2.608088493347168,
  'pred_max_abs': 280029.71875,
  'has_nan': 0,
  'has_inf': 0,
  'available_memory_gb': 52.79230499267578,
  'D_real': 0.0009506612950644922,
  'D_fake': 0.0009156855266025689,
  'D_real_score': 0.7308451574189322,
  'D_fake_score': 0.5002075669595173,
  'D_loss': 0.0009331734133801157,
  'G_adv_raw': 1.0006948564733777,
  'G_struct_raw': 0.9820561853476933,
  'G_struct_unnormalized_raw': 4340.455823451451,
  'G_struct_scale': 4393.196451939174,
  'G_carrier_raw': 6098.503104422433,
  'G_adv_weighted': 1.0006948564733777,
  'G_struct_weighted': 0.9820561853476933,
  'G_carrier_weighted': 3049.2515522112167,
  'G_total': 3051.234298793248},
 'checkpoints': ['02_train/checkpoints/checkpoint_epo

# Cell6 损失曲线

根据 stability.csv 生成/刷新 loss 曲线。

In [6]:
cell6_report = train.cell6_loss_curves(train_result['stability_rows'])
cell6_report

{'loss_curve': '02_train/figures/phase2a_loss_curves.png'}

# Cell7 清理

释放 lock 和 CUDA cache。

In [7]:
cell7_report = train.cell7_cleanup()
cell7_report

{'status': 'cleanup_done', 'timestamp': '2026-06-08 06:58:32 +0800'}

# Cell8 自动摘要

生成 phase2a_summary.md；只报健康事实，不做质量结论。

In [8]:
cell8_report = train.cell8_auto_summary(config)
cell8_report

{'summary_path': '02_train/phase2a_summary.md'}